# 02 — Data Preprocessing & Optimisation

**Pipeline:**
1. Load raw CSVs (`train_transaction.csv` + `train_identity.csv`)
2. Merge on `TransactionID`
3. Drop high-missingness / zero-variance columns
4. Encode categorical columns as `category` dtype
5. Stratified train / val / test split (60 / 20 / 20)
6. Downcast numerical columns to `float32`
7. Save all splits to `data/processed/` as pickles


In [ ]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split

# ── Resolve paths regardless of CWD (notebook root or project root) ──────────
def resolve(rel_nb, rel_root):
    p = Path(rel_nb)
    return p if p.exists() else Path(rel_root)

raw_dir       = resolve("../data/raw",       "data/raw")
processed_dir = resolve("../data/processed", "data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"raw_dir       → {raw_dir.resolve()}")
print(f"processed_dir → {processed_dir.resolve()}")

## 1 · Load Raw CSVs

In [ ]:
print("Loading train_transaction.csv …")
df_txn  = pd.read_csv(raw_dir / "train_transaction.csv")
print(f"  transaction shape : {df_txn.shape}")

print("Loading train_identity.csv …")
df_id   = pd.read_csv(raw_dir / "train_identity.csv")
print(f"  identity shape    : {df_id.shape}")

## 2 · Merge on TransactionID

In [ ]:
df = df_txn.merge(df_id, on="TransactionID", how="left")
print(f"Merged shape: {df.shape}")

# Free raw frames immediately to save RAM
del df_txn, df_id
gc.collect()
print("Raw frames freed from memory.")

## 3 · Drop High-Missingness & Constant Columns

In [ ]:
MISSING_THRESHOLD = 0.9   # drop columns with > 90 % missing values

missing_frac = df.isnull().mean()
cols_to_drop_missing = missing_frac[missing_frac > MISSING_THRESHOLD].index.tolist()
print(f"Columns dropped (>{MISSING_THRESHOLD*100:.0f}% missing): {len(cols_to_drop_missing)}")

# Drop zero-variance (constant) columns
nunique = df.nunique()
cols_constant = nunique[nunique <= 1].index.tolist()
print(f"Columns dropped (constant / zero variance)          : {len(cols_constant)}")

cols_to_drop = list(set(cols_to_drop_missing + cols_constant))
df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping: {df.shape}")

## 4 · Encode Categorical Columns

In [ ]:
# Known categorical columns in the IEEE-CIS dataset
cat_cols = [
    "ProductCD", "card4", "card6",
    "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    "DeviceType", "DeviceInfo",
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28",
    "id_29", "id_30", "id_31", "id_33", "id_34", "id_35",
    "id_36", "id_37", "id_38",
]

# Only convert columns that actually exist after the drop step
existing_cat = [c for c in cat_cols if c in df.columns]
df[existing_cat] = df[existing_cat].astype("category")
print(f"Encoded {len(existing_cat)} categorical columns as 'category' dtype.")

## 5 · Separate Target & Features, then Stratified Split (60 / 20 / 20)

In [ ]:
TARGET = "isFraud"
drop_from_features = [TARGET, "TransactionID"]   # TransactionID is an identifier, not a feature

feature_cols = [c for c in df.columns if c not in drop_from_features]
X = df[feature_cols]
y = df[TARGET].astype(np.int8)

del df
gc.collect()

print(f"Features : {X.shape[1]} columns, {X.shape[0]:,} rows")
print(f"Target   : {y.value_counts().to_dict()}")

# ── First split: 60 % train, 40 % temp ───────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

# ── Second split: 50 % of temp → val (20 % total), 50 % → test (20 % total) ─
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

del X, y, X_temp, y_temp
gc.collect()

print(f"\nSplit summary:")
print(f"  X_train : {X_train.shape}  |  fraud rate: {y_train.mean()*100:.2f}%")
print(f"  X_val   : {X_val.shape}    |  fraud rate: {y_val.mean()*100:.2f}%")
print(f"  X_test  : {X_test.shape}   |  fraud rate: {y_test.mean()*100:.2f}%")

## 6 · Downcast Numerical Columns to `float32`

In [ ]:
# Reduce memory usage of numerical columns
numeric_cols = X_train.select_dtypes(include=["number"]).columns

X_train[numeric_cols] = X_train[numeric_cols].astype("float32")
X_val[numeric_cols]   = X_val[numeric_cols].astype("float32")
X_test[numeric_cols]  = X_test[numeric_cols].astype("float32")

# Note: Categorical columns (ProductCD, card4, card6, P_emaildomain,
# DeviceType, DeviceInfo, …) remain as 'category' dtype — intact.
print(f"Downcasted {len(numeric_cols)} numeric columns to float32.")
print(f"X_train memory : {X_train.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"X_val memory   : {X_val.memory_usage(deep=True).sum()   / 1e6:.1f} MB")
print(f"X_test memory  : {X_test.memory_usage(deep=True).sum()  / 1e6:.1f} MB")

## 7 · Save Processed Splits to `data/processed/`

In [ ]:
# Save optimised datasets to pickles
X_train.to_pickle(processed_dir / "X_train.pkl")
X_val.to_pickle(processed_dir   / "X_val.pkl")
X_test.to_pickle(processed_dir  / "X_test.pkl")

y_train.to_pickle(processed_dir / "y_train.pkl")
y_val.to_pickle(processed_dir   / "y_val.pkl")
y_test.to_pickle(processed_dir  / "y_test.pkl")

print("All processed float32 datasets saved to data/processed/ ✓")
print("Files written:")
for f in sorted(processed_dir.glob("*.pkl")):
    print(f"  {f.name:20s}  {f.stat().st_size / 1e6:.1f} MB")

## 8 · Quick Sanity Check

In [ ]:
# Reload one split to verify integrity
X_train_check = pd.read_pickle(processed_dir / "X_train.pkl")
y_train_check = pd.read_pickle(processed_dir / "y_train.pkl")

assert X_train_check.shape[0] == y_train_check.shape[0], "Row count mismatch!"
assert X_train_check.select_dtypes(include=["float64"]).shape[1] == 0, "float64 columns still present!"

print(f"X_train shape : {X_train_check.shape}")
print(f"y_train shape : {y_train_check.shape}")
print(f"dtypes sample :\n{X_train_check.dtypes.value_counts()}")
print("\n✓ Sanity check passed — preprocessing complete.")